#### # 1. Data Ingestion & Base Staging Setup

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

# Extract data from Bronze staging layer
bronze_df = spark.table("workspace.bronze.erp_loc_a101")

# Staging step to maintain structural architecture consistency 
standardized_df = bronze_df

#### # 2. Business Rules & Location Cleansing Logic

In [0]:
# Apply business rule logic for stripping ID formatting and normalizing country codes
transformed_df = (
    standardized_df
    # 1. Identifiers: Remove dashes from customer ID string
    .withColumn(
        "cid",
        F.regexp_replace(F.col("cid"), "-", "")
    )
    # 2. Attributes (Geography): Standardize country text configurations and handle empty strings
    .withColumn(
        "cntry",
        F.when(F.trim(F.col("cntry")) == "DE", F.lit("Germany"))
        .when(F.trim(F.col("cntry")).isin("US", "USA"), F.lit("United States"))
        .when((F.trim(F.col("cntry")) == "") | F.col("cntry").isNull(), F.lit("N/A"))
        .otherwise(F.trim(F.col("cntry")))
    )
    # 3. Audit Metadata: Operational tracking timestamp
    .withColumn("dwh_create_date", F.current_timestamp())
)

#### # 3. Final Schema Formatting, Renaming, and Target Storage

In [0]:
# Grouping DDL casting and structural organization together (Ordered Sequence)
final_df = transformed_df.select(
    F.col("cid").cast("string"),            # Primary Key / Identifier
    F.col("cntry").cast("string"),          # Geography - Country Name
    F.col("dwh_create_date")                # Warehouse Audit Metadata
)

# Reference mapping ordered exactly to match the selection sequence above
RENAME_MAP = {
    "cid": "customer_id",
    "cntry": "country"
}

renamed_df = final_df
for old_name, new_name in RENAME_MAP.items():
    renamed_df = renamed_df.withColumnRenamed(old_name, new_name)

# Write output schema directly to Silver Delta layer (truncates via overwrite)
renamed_df.write \
    .mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.erp_location")

# Display organized sample preview rows interactively
renamed_df.display()